# 09 — Estimate models and save a portable result bundle

This notebook fits the final founding and survival models using the samples defined in `model_workflow.py`. It writes portable CSV/JSON artifacts to `ANAL/data/models/`; notebook 10 loads those artifacts without rebuilding the model data.

The founding optimizer is initialized from the nested offset model. The unrestricted model must attain at least the restricted model's log-likelihood; execution stops if it does not.


## Configuration


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import platform
import sys

import numpy as np
import pandas as pd
import scipy
import statsmodels
from scipy import stats


def discover_project_dir() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ANAL").is_dir() and (candidate / "OGD").is_dir():
            return candidate
    raise RuntimeError("Run from the project directory or one of its subdirectories.")


PROJECT_DIR = discover_project_dir()
if str(PROJECT_DIR / "ANAL") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "ANAL"))
import model_workflow as mw

PATHS = mw.project_paths(PROJECT_DIR)
RESULT_DIR = PATHS.models
RESULT_DIR.mkdir(parents=True, exist_ok=True)


REBUILD_SECTOR_CACHE = False
MIN_SECTOR_BIRTHS = 500
MIN_KM_EVENTS = 30
MIN_H4_SECTOR_EVENTS = 30




## 1. Founding NB2


In [ ]:
founding, X, y, founding_clusters, founding_terms, ring_specs = mw.founding_model_data(PATHS)
print(f"Founding design: {X.shape[0]:,} rows x {X.shape[1]} columns")


### Fit strategy and hard optimizer check

The offset model fixes the `log_own_firms` coefficient at one and is therefore nested in the unrestricted model. Its estimates provide a stable starting point. The unrestricted likelihood cannot be lower at a valid optimum.


In [ ]:
from statsmodels.discrete.discrete_model import NegativeBinomial, Poisson

X_offset = X.drop(columns=["log_own_firms"])
offset = founding["log_own_firms"].to_numpy(dtype="float64")

poisson_offset = Poisson(y, X_offset, offset=offset).fit(maxiter=200, disp=0)
nb_offset = NegativeBinomial(y, X_offset, loglike_method="nb2", offset=offset).fit(
    start_params=np.append(poisson_offset.params.to_numpy(), 0.1), maxiter=500, disp=0
)

start_main = pd.Series(index=[*X.columns, "alpha"], dtype="float64")
start_main.loc[X_offset.columns] = nb_offset.params.loc[X_offset.columns]
start_main.loc["log_own_firms"] = 1.0
start_main.loc["alpha"] = nb_offset.params.loc["alpha"]

poisson_result = Poisson(y, X).fit(maxiter=200, disp=0)
nb_result = NegativeBinomial(y, X, loglike_method="nb2").fit(
    start_params=start_main.to_numpy(),
    cov_type="cluster", cov_kwds={"groups": founding_clusters},
    maxiter=500, disp=1,
)

if not bool(nb_result.mle_retvals.get("converged", False)):
    raise RuntimeError(f"The founding NB2 optimizer did not converge: {nb_result.mle_retvals}")
if nb_result.llf + 1e-6 < nb_offset.llf:
    raise RuntimeError(
        f"Invalid NB2 optimum: unrestricted LL {nb_result.llf:.3f} is below "
        f"nested offset LL {nb_offset.llf:.3f}."
    )

lr_statistic = max(0.0, 2 * (nb_result.llf - poisson_result.llf))
lr_p_value = 0.5 * stats.chi2.sf(lr_statistic, 1)
print(f"Unrestricted LL: {nb_result.llf:,.2f}")
print(f"Nested offset LL: {nb_offset.llf:,.2f}")
print(f"Boundary LR test alpha=0: statistic={lr_statistic:,.2f}, p={lr_p_value:.4g}")


### Export founding results and robustness summaries


In [ ]:
confidence = nb_result.conf_int()
founding_results = pd.DataFrame({
    "coefficient": nb_result.params,
    "std_error": nb_result.bse,
    "p_value": nb_result.pvalues,
    "ci_lower": confidence.iloc[:, 0],
    "ci_upper": confidence.iloc[:, 1],
})
founding_results["irr"] = np.exp(founding_results["coefficient"])
founding_results["irr_ci_lower"] = np.exp(founding_results["ci_lower"])
founding_results["irr_ci_upper"] = np.exp(founding_results["ci_upper"])
founding_results.loc["alpha", ["irr", "irr_ci_lower", "irr_ci_upper"]] = np.nan
founding_results.to_csv(RESULT_DIR / "founding_nb2_all_terms.csv", index_label="term")
founding_results.loc[
    ~founding_results.index.str.startswith("period_")
].to_csv(RESULT_DIR / "founding_nb2_results.csv", index_label="term")

hessian_covariance = -np.linalg.inv(nb_result.model.hessian(np.asarray(nb_result.params)))
naive_bse = pd.Series(np.sqrt(np.clip(np.diag(hessian_covariance), 0, None)), index=nb_result.params.index)
se_comparison = pd.DataFrame({
    "naive_std_error": naive_bse.reindex(founding_terms),
    "clustered_std_error": nb_result.bse.reindex(founding_terms),
})
se_comparison["clustered_to_naive"] = se_comparison["clustered_std_error"] / se_comparison["naive_std_error"]
se_comparison.to_csv(RESULT_DIR / "founding_nb2_se_comparison.csv", index_label="term")

covariance = np.asarray(nb_result.cov_params())
parameter_names = list(nb_result.params.index)
def linear_contrast(weights):
    vector = np.zeros(len(parameter_names))
    for term, weight in weights.items():
        vector[parameter_names.index(term)] = weight
    estimate = float(vector @ nb_result.params.to_numpy())
    standard_error = float(np.sqrt(vector @ covariance @ vector))
    return estimate, standard_error

decomposition_rows = []
for mass, _, relative_density in ring_specs:
    mode = "walk" if "walk" in mass else "car"
    for effect, weights in {
        "population_holding_firms_fixed": {mass: 1.0, relative_density: -1.0},
        "firms_holding_population_fixed": {relative_density: 1.0},
        "population_and_firms_proportional": {mass: 1.0},
    }.items():
        estimate, standard_error = linear_contrast(weights)
        decomposition_rows.append({
            "mode": mode, "effect": effect, "coefficient": estimate,
            "std_error": standard_error, "irr": np.exp(estimate),
            "irr_ci_lower": np.exp(estimate - 1.96 * standard_error),
            "irr_ci_upper": np.exp(estimate + 1.96 * standard_error),
            "percent_for_10_percent_change": (np.exp(estimate * np.log(1.1)) - 1) * 100,
        })
pd.DataFrame(decomposition_rows).to_csv(RESULT_DIR / "founding_nb2_decomposition.csv", index=False)

growth_sd = float(founding["population_growth_yoy"].std())
growth_beta = float(nb_result.params["population_growth_yoy"])
pt_lines_beta = float(nb_result.params["walk_pt_routes_10min"])
pt_none_beta = float(nb_result.params["pt_ohne_haltestelle"])
diagnostics = pd.Series({
    "n_observations": len(founding),
    "n_cells": founding['grid_id'].nunique(),
    "n_quarters": founding['period'].nunique(),
    "n_births": int(y.sum()),
    "zero_share": float((y == 0).mean()),
    "alpha": float(nb_result.params["alpha"]),
    "converged": bool(nb_result.mle_retvals.get("converged", False)),
    "log_likelihood": float(nb_result.llf),
    "offset_log_likelihood": float(nb_offset.llf),
    "aic": float(nb_result.aic),
    "offset_aic": float(nb_offset.aic),
    "boundary_lr_statistic": lr_statistic,
    "boundary_lr_p_value": lr_p_value,
    "population_growth_sd": growth_sd,
    "birth_change_for_one_sd_growth_percent": (np.exp(growth_beta * growth_sd) - 1) * 100,
    "pt_no_stop_to_stop_percent": (np.exp(-pt_none_beta) - 1) * 100,
    "pt_each_additional_route_percent": (np.exp(pt_lines_beta) - 1) * 100,
    "pt_equal_prediction_route_count": pt_none_beta / pt_lines_beta if pt_lines_beta else np.nan,
})
diagnostics.to_csv(RESULT_DIR / "founding_nb2_diagnostics.csv", header=["value"])

mu = nb_result.predict()
alpha_hat = float(nb_result.params["alpha"])
size = 1.0 / alpha_hat
probability = size / (size + mu)
categories = [0, 1, 2, 3, 4]
modelled = [float(stats.nbinom.pmf(k, size, probability).mean()) for k in categories]
modelled.append(1.0 - sum(modelled))
observed = [float((y == k).mean()) for k in categories]
observed.append(float((y >= 5).mean()))
pd.DataFrame({
    "births": [*map(str, categories), "5+"], "observed_share": observed, "modelled_share": modelled
}).to_csv(RESULT_DIR / "founding_nb2_count_fit.csv", index=False)
display(founding_results.loc[founding_terms].round(4))


### Optional H4 founding models by official sector


In [ ]:
access_path, births_path, stock_path, sector_mapping = mw.prepare_founding_sector_cache(
    PATHS, rebuild=REBUILD_SECTOR_CACHE
)
sector_births = pd.read_parquet(births_path).groupby("sparte")["births_sparte"].sum()
sectors = [str(s) for s, count in sector_births.items() if count >= MIN_SECTOR_BIRTHS]
sector_result_rows = []
sector_diagnostic_rows = []
sector_population_rows = []
sector_population_influence = {}
for sector in sorted(sectors):
    X_sector, y_sector, clusters_sector, terms_sector = mw.founding_sector_design(
        founding, access_path, births_path, stock_path, sector
    )
    poisson_sector = Poisson(y_sector, X_sector).fit(maxiter=200, disp=0)
    nb_sector = NegativeBinomial(y_sector, X_sector, loglike_method="nb2").fit(
        start_params=np.append(poisson_sector.params.to_numpy(), 0.1),
        cov_type="cluster", cov_kwds={"groups": clusters_sector}, maxiter=500, disp=0,
    )
    if not bool(nb_sector.mle_retvals.get("converged", False)):
        raise RuntimeError(f"Sector {sector} founding model did not converge.")
    ci = nb_sector.conf_int()
    sector_name = sector_mapping.loc[sector_mapping["sparte"] == sector, "sparte_name"].iloc[0]
    parameter_names = list(nb_sector.params.index)
    population_contrast = np.zeros(len(parameter_names))
    population_contrast[parameter_names.index("log_pop_access_ring_0_15")] = 1.0
    population_contrast[parameter_names.index("log_same_relative_car_ring_0_15")] = -1.0
    population_contrast[parameter_names.index("log_other_relative_car_ring_0_15")] = -1.0
    population_effect = float(population_contrast @ nb_sector.params.to_numpy())
    population_se = float(np.sqrt(population_contrast @ np.asarray(nb_sector.cov_params()) @ population_contrast))
    sector_population_rows.append({
        "sparte": sector, "sparte_name": sector_name, "coefficient": population_effect,
        "std_error": population_se, "irr": np.exp(population_effect),
        "irr_ci_lower": np.exp(population_effect - 1.96 * population_se),
        "irr_ci_upper": np.exp(population_effect + 1.96 * population_se),
    })
    score_observations = nb_sector.model.score_obs(np.asarray(nb_sector.params))
    cluster_codes, cluster_labels = pd.factorize(clusters_sector, sort=False)
    grouped_scores = np.zeros((len(cluster_labels), score_observations.shape[1]))
    np.add.at(grouped_scores, cluster_codes, score_observations)
    bread = np.linalg.inv(nb_sector.model.hessian(np.asarray(nb_sector.params)))
    sector_population_influence[sector] = pd.Series(
        population_contrast @ bread @ grouped_scores.T, index=cluster_labels
    )
    for term in terms_sector:
        sector_result_rows.append({
            "sparte": sector, "sparte_name": sector_name, "term": term,
            "coefficient": nb_sector.params[term], "std_error": nb_sector.bse[term],
            "irr": np.exp(nb_sector.params[term]),
            "irr_ci_lower": np.exp(ci.loc[term, 0]), "irr_ci_upper": np.exp(ci.loc[term, 1]),
        })
    sector_diagnostic_rows.append({
        "sparte": sector, "sparte_name": sector_name, "n_births": int(y_sector.sum()),
        "alpha": float(nb_sector.params["alpha"]), "log_likelihood": float(nb_sector.llf),
        "converged": bool(nb_sector.mle_retvals.get("converged", False)),
    })
    print(f"Sector {sector} ({sector_name}): {int(y_sector.sum()):,} births")
pd.DataFrame(sector_result_rows).to_csv(RESULT_DIR / "founding_nb2_sector_results.csv", index=False)
pd.DataFrame(sector_diagnostic_rows).to_csv(RESULT_DIR / "founding_nb2_sector_diagnostics.csv", index=False)
sector_population = pd.DataFrame(sector_population_rows)
sector_population.to_csv(RESULT_DIR / "founding_sector_population_effects.csv", index=False)
if len(sector_population) > 1:
    influence = pd.DataFrame(sector_population_influence).fillna(0.0)
    cross_model_covariance = influence.to_numpy().T @ influence.to_numpy()
    estimates = sector_population.set_index("sparte").loc[influence.columns, "coefficient"].to_numpy()
    contrast = np.column_stack([-np.ones(len(estimates) - 1), np.eye(len(estimates) - 1)])
    differences = contrast @ estimates
    difference_covariance = contrast @ cross_model_covariance @ contrast.T
    rank = int(np.linalg.matrix_rank(difference_covariance))
    statistic = float(differences @ np.linalg.pinv(difference_covariance) @ differences)
    pd.Series({
        "effect": "reachable_population_holding_same_and_other_firms_fixed",
        "wald_chi2": statistic, "degrees_of_freedom": rank,
        "p_value": float(stats.chi2.sf(statistic, rank)) if rank else np.nan,
        "covariance": "cross-model cell-clustered sandwich (CR0)",
    }).to_csv(RESULT_DIR / "founding_sector_joint_test.csv", header=["value"])


## 2. Fachgruppe-stratified Cox survival model


In [ ]:
from statsmodels.duration.hazard_regression import PHReg

spells, survival_frame, survival_terms = mw.survival_model_data(PATHS)
cox_model = PHReg(
    endog=survival_frame["stop"].to_numpy(dtype="float64"),
    exog=survival_frame[survival_terms].astype("float64"),
    status=survival_frame["event"].to_numpy(dtype="float64"),
    entry=survival_frame["start"].to_numpy(dtype="float64"),
    strata=survival_frame["Fachgruppe_ID"].to_numpy(),
    ties="efron",
)
cox_result = cox_model.fit()
cluster_cov, rows_outside_risk_sets = mw.cluster_covariance(
    cox_model, np.asarray(cox_result.params), survival_frame["grid_id"].to_numpy()
)
cluster_variances = np.diag(cluster_cov)
if (cluster_variances < -1e-12).any():
    raise RuntimeError("Cluster covariance has a negative diagonal element.")
cluster_bse = np.sqrt(np.clip(cluster_variances, 0, None))
print(f"Survival intervals: {len(survival_frame):,}; exits: {int(survival_frame['event'].sum()):,}")


### Export Cox estimates and proportional-hazards screen


In [ ]:
parameter = np.asarray(cox_result.params)
critical = stats.norm.ppf(0.975)
z_values = parameter / cluster_bse
survival_results = pd.DataFrame({
    "coefficient": parameter,
    "std_error": cluster_bse,
    "z": z_values,
    "p_value": 2 * stats.norm.sf(np.abs(z_values)),
    "hazard_ratio": np.exp(parameter),
    "hr_ci_lower": np.exp(parameter - critical * cluster_bse),
    "hr_ci_upper": np.exp(parameter + critical * cluster_bse),
}, index=survival_terms)
survival_results.to_csv(RESULT_DIR / "survival_cox_results.csv", index_label="term")

survival_contrasts = {
    "population_holding_same_and_other_firms_fixed": {
        "log_pop_ring_0_15": 1.0, "log_same_relative_ring_0_15": -1.0,
        "log_other_relative_ring_0_15": -1.0,
    },
    "same_group_firms_holding_population_fixed": {"log_same_relative_ring_0_15": 1.0},
    "other_firms_holding_population_fixed": {"log_other_relative_ring_0_15": 1.0},
    "population_and_both_firm_stocks_proportional": {"log_pop_ring_0_15": 1.0},
}
survival_decomposition_rows = []
for effect, weights in survival_contrasts.items():
    vector = np.zeros(len(survival_terms))
    for term, weight in weights.items():
        vector[survival_terms.index(term)] = weight
    estimate = float(vector @ parameter)
    standard_error = float(np.sqrt(vector @ cluster_cov @ vector))
    survival_decomposition_rows.append({
        "effect": effect, "coefficient": estimate, "std_error": standard_error,
        "hazard_ratio": np.exp(estimate),
        "hr_ci_lower": np.exp(estimate - critical * standard_error),
        "hr_ci_upper": np.exp(estimate + critical * standard_error),
    })
pd.DataFrame(survival_decomposition_rows).to_csv(
    RESULT_DIR / "survival_cox_decomposition.csv", index=False
)

schoenfeld_residuals = np.asarray(cox_result.schoenfeld_residuals)
event_rows = ~np.isnan(schoenfeld_residuals).any(axis=1)
event_age = survival_frame.loc[event_rows, "stop"].to_numpy(dtype="float64") / 4
ranked_age = stats.rankdata(event_age)
ph_rows = []
for column, term in enumerate(survival_terms):
    rho, p_value = stats.spearmanr(ranked_age, schoenfeld_residuals[event_rows, column])
    ph_rows.append({"term": term, "spearman_rho": rho, "screen_p_value": p_value})
pd.DataFrame(ph_rows).to_csv(RESULT_DIR / "survival_schoenfeld_screen.csv", index=False)

cluster_sizes = survival_frame.groupby("grid_id").size()
survival_diagnostics = pd.Series({
    "n_intervals": len(survival_frame),
    "n_locations": survival_frame['standort_id'].nunique(),
    "n_events": int(survival_frame['event'].sum()),
    "n_cells": survival_frame['grid_id'].nunique(),
    "n_fachgruppe_strata": survival_frame['Fachgruppe_ID'].nunique(),
    "rows_outside_event_risk_sets": rows_outside_risk_sets,
    "rows_outside_event_risk_sets_share": rows_outside_risk_sets / len(survival_frame),
    "effective_cluster_size": float(cluster_sizes.pow(2).sum() / cluster_sizes.sum()),
})
survival_diagnostics.to_csv(RESULT_DIR / "survival_cox_diagnostics.csv", header=["value"])
display(survival_results.round(4))


### Export read-only visualization data


In [ ]:
from statsmodels.duration.survfunc import SurvfuncRight

location_spells = (
    survival_frame.groupby("standort_id", sort=False)
    .agg(sparte=("sparte", "first"), sparte_name=("sparte_name", "first"),
         entry=("start", "min"), exit=("stop", "max"), event=("event", "max"))
    .reset_index()
)
km_rows = []
for sector, group in location_spells.groupby("sparte"):
    if group["event"].sum() < MIN_KM_EVENTS:
        continue
    curve = SurvfuncRight(
        group["exit"].to_numpy(dtype="float64") / 4,
        group["event"].to_numpy(dtype="float64"),
        entry=group["entry"].to_numpy(dtype="float64") / 4,
    )
    sector_name = group["sparte_name"].iloc[0]
    km_rows.extend({
        "sparte": sector, "sparte_name": sector_name,
        "time_years": time, "survival_probability": probability,
        "n_locations": len(group), "n_events": int(group['event'].sum()),
    } for time, probability in zip(curve.surv_times, curve.surv_prob))
pd.DataFrame(km_rows).to_csv(RESULT_DIR / "survival_km_by_sector.csv", index=False)

annual_exit_rates = (
    survival_frame.groupby(["sparte", "sparte_name", "year"])
    .agg(exits=("event", "sum"), locations_at_risk=("standort_id", "nunique"))
    .reset_index()
)
annual_exit_rates["exit_rate"] = annual_exit_rates["exits"] / annual_exit_rates["locations_at_risk"]
annual_exit_rates.to_csv(RESULT_DIR / "survival_annual_exit_rates.csv", index=False)


### Optional H4 survival interaction by official sector


In [ ]:
H4_BASIS = "log_own_same"
h4_support = survival_frame.groupby("sparte").agg(
    events=("event", "sum"), basis_variation=(H4_BASIS, "nunique")
)
sectors = sorted(h4_support.index[
    (h4_support["events"] >= MIN_H4_SECTOR_EVENTS) & (h4_support["basis_variation"] > 1)
])
if len(sectors) < 2:
    raise RuntimeError(f"Fewer than two sectors support the H4 interaction:\n{h4_support}")
reference_sector = sectors[0]
interaction_terms = []
survival_h4 = survival_frame.loc[survival_frame["sparte"].isin(sectors)].copy()
event_strata = survival_h4.groupby("Fachgruppe_ID")["event"].sum()
survival_h4 = survival_h4.loc[survival_h4["Fachgruppe_ID"].isin(event_strata[event_strata > 0].index)].copy()
for sector in sectors[1:]:
    term = f"{H4_BASIS}_x_sparte_{sector}"
    survival_h4[term] = survival_h4[H4_BASIS] * survival_h4["sparte"].eq(sector).astype("float64")
    interaction_terms.append(term)
h4_terms = [*survival_terms, *interaction_terms]
h4_model = PHReg(
    endog=survival_h4["stop"].to_numpy(dtype="float64"),
    exog=survival_h4[h4_terms].astype("float64"),
    status=survival_h4["event"].to_numpy(dtype="float64"),
    entry=survival_h4["start"].to_numpy(dtype="float64"),
    strata=survival_h4["Fachgruppe_ID"].to_numpy(), ties="efron",
)
h4_result = h4_model.fit()
h4_cov, _ = mw.cluster_covariance(h4_model, np.asarray(h4_result.params), survival_h4["grid_id"].to_numpy())
h4_parameter = np.asarray(h4_result.params)
sector_effect_rows = []
for sector in sectors:
    vector = np.zeros(len(h4_terms))
    vector[h4_terms.index(H4_BASIS)] = 1.0
    if sector != reference_sector:
        vector[h4_terms.index(f"{H4_BASIS}_x_sparte_{sector}")] = 1.0
    estimate = float(vector @ h4_parameter)
    standard_error = float(np.sqrt(vector @ h4_cov @ vector))
    name = survival_h4.loc[survival_h4["sparte"] == sector, "sparte_name"].iloc[0]
    sector_effect_rows.append({
        "sparte": sector, "sparte_name": name, "basis_term": H4_BASIS,
        "coefficient": estimate, "std_error": standard_error,
        "hazard_ratio": np.exp(estimate),
        "hr_ci_lower": np.exp(estimate - 1.96 * standard_error),
        "hr_ci_upper": np.exp(estimate + 1.96 * standard_error),
    })
pd.DataFrame(sector_effect_rows).to_csv(RESULT_DIR / "survival_sector_effects.csv", index=False)

contrast = np.zeros((len(interaction_terms), len(h4_terms)))
for row, term in enumerate(interaction_terms):
    contrast[row, h4_terms.index(term)] = 1.0
difference = contrast @ h4_parameter
contrast_cov = contrast @ h4_cov @ contrast.T
statistic = float(difference @ np.linalg.pinv(contrast_cov) @ difference)
rank = int(np.linalg.matrix_rank(contrast_cov))
pd.Series({
    "basis_term": H4_BASIS, "reference_sector": reference_sector,
    "wald_chi2": statistic, "degrees_of_freedom": rank,
    "p_value": float(stats.chi2.sf(statistic, rank)) if rank else np.nan,
}).to_csv(RESULT_DIR / "survival_sector_joint_test.csv", header=["value"])


## 3. Write run manifest


In [ ]:
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_period": {"start_year": mw.START_YEAR, "end_year": mw.END_YEAR, "founding_lag_year": mw.LAG_YEAR},
    "software": {
        "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
        "scipy": scipy.__version__, "statsmodels": statsmodels.__version__,
    },
}
with open(RESULT_DIR / "model_run_manifest.json", "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

print(f"Portable result bundle written to: {RESULT_DIR}")
print("Copy this directory—not the raw data—to the visualization machine.")
